In [1]:
import numpy as np
import time
import pandas as pd
from rail_rl_env import RailNet, gurobi_minlp, gurobi_qp, qp_feasible

data_sets = np.load('training_sets.npy', allow_pickle=True).item()
d_pre = data_sets['d_pre']
rho_whole = data_sets['rho_whole']
un = data_sets['un']
ul = data_sets['ul']
uy = data_sets['uy']
ua = data_sets['ua']
ud = data_sets['ud']
utau = data_sets['utau']
ur = data_sets['ur']
depot = data_sets['depot']

r_max = data_sets['r_max']
r_min = data_sets['r_min']
differ = data_sets['differ']
Cmax = data_sets['Cmax']
sigma = data_sets['sigma']
same = data_sets['same']
num_station = data_sets['num_station']
num_train = data_sets['num_train']
E_regular = data_sets['E_regular']

control_trains = 8
N = int(control_trains)
epsilon = 10 ** (-10)
Mt = 1000000
mt = -1000000
t_constant = 60
h_min = 120
tau_min = 30
l_min = 1
l_max = 4
eta = 10**(-3)

g = 3

Env = RailNet(N)

In [2]:
from rail_rl_env import action_dict

inv_action_dict = {tuple(v[0]): int(k) for k, v in action_dict.items()}

def build_list_action(delta: np.array) -> list:
    delta = np.round(delta,2)
    list_action = [inv_action_dict[tuple(delta[0])]]
    for i in range(1, control_trains-2):
        list_action.append(inv_action_dict[tuple(delta[i])])
        
    return list_action

In [3]:
control_trains = 8
N = int(control_trains)
Env = RailNet(N)

total_time_minlp = []
total_time_nlp = []
total_time_nlp2 = []
runtime_minlp = []
runtime_nlp = []
runtime_nlp2 = []

cntr_feasible = 0

for i in range(10):
    
    print(i)

    Env.set_randState(d_pre, rho_whole, un, ul, uy, ua, ud, ur, depot)

    start_time_minlp = time.time()
    a_minlp, d_minlp, r_minlp, l_minlp, y_minlp, delta_minlp, mdl_minlp = gurobi_minlp(control_trains, Env.d_pre_cut, Env.state_rho,
                            Env.state_a, Env.state_d, Env.state_r, Env.state_l, Env.state_y, Env.state_n, Env.state_depot,
                            num_station, differ, sigma, same, t_constant, h_min, l_min, l_max, r_min, r_max, tau_min,
                            E_regular, Cmax, eta)
    end_time_minlp = time.time()
    total_time_minlp.append(end_time_minlp-start_time_minlp)
    runtime_minlp.append(mdl_minlp.Runtime)

    # delta = delta_minlp.astype(np.int32)
    delta = delta_minlp

    list_action = build_list_action(delta)

    start_time_nlp = time.time()
    a_minlp2, d_minlp2, r_minlp2, l_minlp2, y_minlp2, mdl_minlp2 = gurobi_qp(control_trains, Env.d_pre_cut, Env.state_rho,
                            Env.state_a, Env.state_d, Env.state_r, Env.state_l, Env.state_y, Env.state_n, Env.state_depot, delta,
                            num_station, differ, sigma, same, t_constant, h_min, l_min, l_max, r_min, r_max, tau_min,
                            E_regular, Cmax, eta)
    end_time_nlp = time.time()
    total_time_nlp.append(end_time_nlp-start_time_nlp)
    runtime_nlp.append(mdl_minlp2.Runtime)
    
    if qp_feasible(mdl_minlp):
        cntr_feasible += 1
    
    # start_time_nlp = time.time()
    # a_minlp2, d_minlp2, r_minlp2, l_minlp2, y_minlp2, mdl_minlp2 = gurobi_qp(control_trains, Env.d_pre_cut, Env.state_rho,
    #                         Env.state_a, Env.state_d, Env.state_r, Env.state_l, Env.state_y, Env.state_n, Env.state_depot, delta,
    #                         num_station, differ, sigma, same, t_constant, h_min, l_min, l_max, r_min, r_max, tau_min,
    #                         E_regular, Cmax, eta)
    # end_time_nlp = time.time()
    # total_time_nlp2.append(end_time_nlp-start_time_nlp)
    # runtime_nlp2.append(mdl_minlp2.Runtime)

state_rho, d_pre_cut, state_a, state_d, state_r, state_l, state_y,state_n, state_depot, reward, terminated, truncated, info = Env.step(list_action, d_pre, rho_whole, r_max, r_min, differ, Cmax, sigma, same, num_station, num_train, E_regular)

# Env.set_randState(d_pre, rho_whole, un, ul, uy, ua, ud, ur, depot)

# a_minlp2, d_minlp2, r_minlp2, l_minlp2, y_minlp2, delta_minlp2, mdl_minlp2 = gurobi_minlp(control_trains, Env.d_pre_cut, Env.state_rho,
#                         Env.state_a, Env.state_d, Env.state_r, Env.state_l, Env.state_y, Env.state_n, Env.state_depot,
#                         num_station, differ, sigma, same, t_constant, h_min, l_min, l_max, r_min, r_max, tau_min,
#                         E_regular, Cmax, eta)

# delta2 = delta_minlp2
# list_action2 = build_list_action(delta2)
# state_rho2, d_pre_cut2, state_a2, state_d2, state_r2, state_l2, state_y2, state_n2, state_depot2, reward2, terminated2, truncated2, info2 = Env.step(list_action2, d_pre, rho_whole, r_max, r_min, differ, Cmax, sigma, same, num_station, num_train, E_regular)

# print(mdl_minlp.ObjVal - info['mdl'].ObjVal)
# print(mdl_minlp2.ObjVal - info2['mdl'].ObjVal)

0
Set parameter Username
Academic license - for non-commercial use only - expires 2025-04-08


1
2
3
4
5
6
7
8
9


In [4]:
cntr_feasible

10

In [5]:
solution_minlp = []
N_datapoints = 10

def compress_minlp_info(state_n, state_rho, state_depot, state_l, delta_minlp, mdl_Obj):
    
    minlp_info_compressed = np.concatenate((state_n, state_rho.flatten(), state_depot, state_l.flatten(), delta_minlp.flatten(), mdl_Obj.reshape(1,)))
    
    return minlp_info_compressed

minlp_info_compressed = np.zeros((N_datapoints, 427))

start_time = time.time()

cntr_feasible = 0

for i in range(N_datapoints):

    Env.set_randState(d_pre, rho_whole, un, ul, uy, ua, ud, ur, depot)

    state_n = Env.state_n
    state_rho = Env.state_rho
    state_depot = Env.state_depot
    state_l = Env.state_l

    a_minlp, d_minlp, r_minlp, l_minlp, y_minlp, delta_minlp, mdl_minlp = gurobi_minlp(control_trains, Env.d_pre_cut, Env.state_rho,
                            Env.state_a, Env.state_d, Env.state_r, Env.state_l, Env.state_y, Env.state_n, Env.state_depot,
                            num_station, differ, sigma, same, t_constant, h_min, l_min, l_max, r_min, r_max, tau_min,
                            E_regular, Cmax, eta)
    
    if qp_feasible(mdl_minlp)==True:
        minlp_info_compressed[i,:] = compress_minlp_info(state_n, state_rho, state_depot, state_l, delta_minlp, np.array(mdl_minlp.ObjVal))
        cntr_feasible += 1
    
    elapsed_time = time.time()-start_time
    
    if i == 0:
        mdl_minlp.Params.LogToConsole = 1
        mdl_minlp.optimize()
    
    if i % 100 == 0:
        print('i=%d' %i, 'elapsed_time=%.2f' %elapsed_time, 'avg_solution_time=%.2f' %(elapsed_time/(i+1)))
    
    # print('i=%d' %i, 'elapsed_time=%.2f' %elapsed_time)
    
# np.save('data_minlp//data_minlp_N%.2d.npy' %N, minlp_info_compressed, allow_pickle=True)

print('cntr_feasible = %d' % cntr_feasible)
print('completed')

Set parameter LogToConsole to value 1
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (22631.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13700T, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 24 logical processors, using up to 24 threads

Optimize a model with 6313 rows, 53536 columns and 12017 nonzeros
Model fingerprint: 0x7914b31d
Model has 448 quadratic objective terms
Model has 532 quadratic constraints
Variable types: 2240 continuous, 51296 integer (50624 binary)
Coefficient statistics:
  Matrix range     [4e-03, 4e+02]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+06]
  Objective range  [2e+01, 5e+01]
  QObjective range [2e-03, 2e-03]
  Bounds range     [1e+00, 4e+00]
  RHS range        [1e+00, 3e+04]
  QRHS range       [4e+00, 1e+06]
Presolved: 1403 rows, 1201 columns, 4437 nonzeros
Presolved model has 305 bilinear constraint(s)

Solving non-convex MIQCP


Continuing optimization...


Cutting planes:
  Learned: 1
  MI

In [16]:
np.mean(total_time_minlp), np.mean(total_time_nlp), np.mean(runtime_minlp), np.mean(runtime_nlp)

(3.8040520191192626,
 1.5783387660980224,
 3.4907999992370606,
 1.2635000228881836)

In [17]:
delta

array([[1., 0., 0., 0., 1., 0., 1., 1.],
       [1., 0., 0., 0., 1., 0., 1., 1.],
       [1., 0., 0., 0., 1., 0., 1., 1.],
       [1., 0., 0., 0., 1., 0., 1., 0.],
       [0., 0., 0., 0., 1., 0., 1., 0.],
       [0., 0., 0., 0., 1., 0., 0., 1.]])

In [13]:
qp_feasible(mdl_minlp)

False

In [6]:
mdl_minlp.ObjVal - mdl_minlp2.ObjVal

AttributeError: Unable to retrieve attribute 'ObjVal'

In [15]:
print('state_rho:\t', state_rho.shape)
print('d_pre_cut:\t', d_pre_cut.shape)
print('state_a:\t', state_a.shape)
print('state_d:\t', state_d.shape)
print('state_r:\t', state_r.shape)
print('state_l:\t', state_l.shape)
print('state_y:\t', state_y.shape)
print('state_n:\t', state_n.shape)
print('state_depot:\t', state_depot.shape)

state_rho:	 (9, 28)
d_pre_cut:	 (10, 28)
state_a:	 (10, 28)
state_d:	 (3, 28)
state_r:	 (3, 28)
state_l:	 (3, 28)
state_y:	 (3, 28)
state_n:	 (28,)
state_depot:	 (14,)


In [17]:
idx = 0

d_pre_cut_learning = d_pre_cut[idx,:] - np.min(d_pre_cut)# takes the minimum departure time as the reference (makes it invariant to the time of day)
print(d_pre_cut[idx,:])
print(d_pre_cut_learning)

# d_pre_cut_learning2 = d_pre_cut2[idx,:] - np.min(d_pre_cut2)# takes the minimum departure time as the reference (makes it invariant to the time of day)
# print(d_pre_cut_learning - d_pre_cut_learning2)

[10860.         10750.62352941 10878.44117647 10807.23529412
 10749.05294118 10692.77647059 10824.51176471 10702.21764706
 10853.7        10726.27058824 10891.2        10836.45882353
 10723.95882353 10683.24705882 10796.18823529 10755.47647059
 10882.97647059 10828.23529412 10753.16470588 10865.73529412
 10777.21764706 10894.92352941 10786.65882353 10730.38235294
 10912.2        10840.99411765 10728.81176471 10859.43529412]
[176.75294118  67.37647059 195.19411765 123.98823529  65.80588235
   9.52941176 141.26470588  18.97058824 170.45294118  43.02352941
 207.95294118 153.21176471  40.71176471   0.         112.94117647
  72.22941176 199.72941176 144.98823529  69.91764706 182.48823529
  93.97058824 211.67647059 103.41176471  47.13529412 228.95294118
 157.74705882  45.56470588 176.18823529]


In [ ]:
# tests coherence between gurobi_minlp and gurobi_qp

list_minlp = []
list_qp = []

for i in range(100):

    Env.set_randState(d_pre, rho_whole, un, ul, uy, ua, ud, ur, depot)

    a_minlp, d_minlp, r_minlp, l_minlp, y_minlp, delta_minlp, mdl_minlp = gurobi_minlp(control_trains, Env.d_pre_cut, Env.state_rho,
                        Env.state_a, Env.state_d, Env.state_r, Env.state_l, Env.state_y, Env.state_n, Env.state_depot,
                        num_station, differ, sigma, same, t_constant, h_min, l_min, l_max, r_min, r_max, tau_min,
                        E_regular, Cmax, eta)

    # delta = self.build_delta_vector(list_action)
    '''use delta generated by minlp to guribi_qp'''
    delta = delta_minlp
    a_qp, d_qp, r_qp, l_qp, y_qp, mdl_qp = gurobi_qp(Env.control_trains,Env.d_pre_cut,Env.state_rho,
                    Env.state_a, Env.state_d, Env.state_r, Env.state_l, Env.state_y, Env.state_n,Env.state_depot,
                    delta, num_station,differ,sigma,same,t_constant,h_min,l_min,l_max,r_min,r_max,tau_min,E_regular,Cmax,eta)

    list_minlp.append(mdl_minlp.Obj)
    list_qp.append(mdl_qp.Obj)
list_minlp == list_qp

In [3]:
import datetime

x = datetime.datetime.now()
type(x.month), x.day

(int, 24)